# ทดลองระบบตอบไงดี AI ภาษาไทย

Notebook นี้ใช้ทดลอง rule-based flow และทดลอง train โมเดล ML เบื้องต้นจาก dataset ตัวอย่าง

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.analyze_chat_th import analyze_chat_th

## ทดลองวิเคราะห์ข้อความหลายแบบ

In [2]:
examples = [
    ("ทำไรอยู่", "flirty", "crush"),
    ("กินข้าวยัง", "cute", "friend"),
    ("คิดถึง", "flirty", "crush"),
    ("แล้วแต่", "cold", "friend"),
    ("ขอโทษนะ", "polite", "coworker"),
]

for text, style, relationship in examples:
    result = analyze_chat_th(text, style=style, relationship=relationship)
    print("-" * 40)
    print(f"ข้อความ: {result['input_text']}")
    print(f"intent: {result['intent']}")
    print(f"style: {result['style']}")
    print(f"relationship: {result['relationship']}")
    print("คำตอบแนะนำ:")
    for index, reply in enumerate(result["recommended_replies"], start=1):
        print(f"{index}. {reply}")

----------------------------------------
ข้อความ: ทำไรอยู่
intent: daily_question
style: flirty
relationship: crush
คำตอบแนะนำ:
1. ทำงานอยู่ แต่คุยกับเธอได้
2. ไม่ค่อยได้ทำไร รอคนบางคนทักมา
3. กำลังคิดอยู่ว่าจะทักเธอดีไหมพอดี
----------------------------------------
ข้อความ: กินข้าวยัง
intent: daily_question
style: cute
relationship: friend
คำตอบแนะนำ:
1. กำลังคิดอยู่ว่าจะตอบยังไงให้น่ารักดี
2. กำลังว่างให้เธอกวนพอดี
3. นั่งเล่นอยู่ แต่คุยกับเธอได้เสมอ
----------------------------------------
ข้อความ: คิดถึง
intent: flirt
style: flirty
relationship: crush
คำตอบแนะนำ:
1. พูดแบบนี้จะให้เรายิ้มทั้งวันเลยเหรอ
2. อยากเจอก็มาเจอสิ รออยู่
3. คิดถึงเหมือนกัน มากกว่าที่บอกอีก
----------------------------------------
ข้อความ: แล้วแต่
intent: dry_reply
style: cold
relationship: friend
คำตอบแนะนำ:
1. เค
2. ครับ
3. อืม
----------------------------------------
ข้อความ: ขอโทษนะ
intent: apology
style: polite
relationship: coworker
คำตอบแนะนำ:
1. ขอบคุณที่อธิบายนะครับ
2. เข้าใจค่ะ ไว้คุยกันดี ๆ นะ
3. ไ

## ทดลอง train ML เบื้องต้น

ภาษาไทยไม่มีการเว้นวรรคคำชัดเจน การใช้ char n-gram ช่วยให้โมเดลจับ pattern จากตัวอักษรได้โดยไม่ต้องใช้ตัวตัดคำไทย

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

dataset_path = PROJECT_ROOT / "data" / "th_chat_intent_dataset.csv"
df = pd.read_csv(dataset_path)
df.head()

,text,intent
0,หวัดดี,greeting
1,สวัสดี,greeting
2,ดีจ้า,greeting
3,ทักนะ,greeting
4,ฮัลโหล,greeting


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["intent"],
    test_size=0.25,
    random_state=42,
    stratify=df["intent"],
)

model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 5))),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(classification_report(y_test, predictions, zero_division=0))

                precision    recall  f1-score   support

         angry       0.00      0.00      0.00         2
       apology       1.00      1.00      1.00         2
    ask_status       0.00      0.00      0.00         2
       comfort       1.00      1.00      1.00         2
    compliment       0.67      1.00      0.80         2
daily_question       1.00      1.00      1.00         2
     dry_reply       0.00      0.00      0.00         2
         flirt       0.00      0.00      0.00         2
     goodnight       1.00      1.00      1.00         2
      greeting       0.00      0.00      0.00         2
        invite       1.00      1.00      1.00         2

      accuracy                           0.55        22
     macro avg       0.52      0.55      0.53        22
  weighted avg       0.52      0.55      0.53        22



In [5]:
sample_texts = ["ทำไรอยู่", "น่ารักจัง", "ไม่ต้องยุ่ง", "ฝันดีนะ"]
for text in sample_texts:
    print(text, "->", model.predict([text])[0])

ทำไรอยู่ -> daily_question
น่ารักจัง -> compliment
ไม่ต้องยุ่ง -> angry
ฝันดีนะ -> goodnight
